In [3]:
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, TensorDataset
import matplotlib.pyplot as plt
import pandas as pd
from datetime import datetime
import numpy as np

In [ ]:
# Hyperparameters
layers = [3, 64, 64, 64, 1]
lr = 1e-3
batch_size = 32
epochs = 50

In [ ]:
def sdf_box(p, center, half_extents):
    """
    SDF for an axis-aligned box.

    Parameters
    ----------
    p : (N, 2) array of points to evaluate
    center : (2,) array-like — center of the box e.g. [0.0, 0.0]
    half_extents : (2,) array-like — half the width and height of the box e.g. [0.5, 0.5]
    """
    q = np.abs(p - center) - half_extents
    return np.linalg.norm(np.maximum(q, 0), axis=-1) + np.minimum(np.max(q, axis=-1), 0)


def sdf_t_block(p, bar_width, bar_height, stem_width, stem_height):
    """
    T-shape as union of two boxes.
    Origin is at the center of the full bounding box.

        ┌─────────────┐  ↑
        │   top bar   │  bar_height
        └────┬───┬────┘  ↓
             │ s │  ↑
             │ t │  stem_height
             │ e │
             │ m │  ↓
             └───┘
    """
    bar_center = np.array([0.0, stem_height / 2])
    stem_center = np.array([0.0, -bar_height / 2])

    overlap = 0.05
    bar = sdf_box(p, bar_center, np.array([bar_width / 2, bar_height / 2]))
    stem = sdf_box(
        p, 
        stem_center + np.array([0.0, overlap / 2]), 
        np.array([stem_width / 2, stem_height / 2 + overlap / 2])
    )

    return np.minimum(bar, stem)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Plate parameters
size = 2.0
t_max = 5.0
alpha = 0.01

SDF = lambda coords: sdf_t_block(
    coords, bar_width=1.5, bar_height=0.385, stem_width=0.385, stem_height=1.0
)